# Workflow

This notebook was implemented in Google Colab. 

1. Select AOI (loaded from [gsheet](
https://docs.google.com/spreadsheets/d/1sMJMJD3xOFdhyi2du6cyQdkJ6mooFLCmcsyu6_IAt_o/edit?gid=1985968760#gid=1985968760))(**1**).
2. Visualise rainfall time series. Start with Monthly to Daily timescales to extract maximum precipitation dates. This is to avoid loading large data points **2.1**).
3. Import Sarno Data
4. Execute River Plastic Detection

References:

Muñoz Sabater, J., (2019): ERA5-Land monthly averaged data from 1981 to present. Copernicus Climate Change Service (C3S) Climate Data Store (CDS). (2025-08-12), [doi:10.24381/cds.68d2bb30](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-monthly-means?tab=overview)

# Setup

## GEE Authentication

In [ ]:
#!pip install geemap

In [ ]:
# Import Libraries

import ee
import geemap.foliumap as geemap

In [ ]:
# GEE Authentication

ee.Authenticate()
ee.Initialize(project='#ee-id')

## Google Drive Authentication

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 1-Load AOI

In [ ]:
import pandas as pd

# You can get the file ID from the URL of your Google Sheet
file_id = 'change with google sheet file details'
sheet_name = 'Sarno_Events' #Events #Sarno_Events Replace with the name of your sheet if it's not Sheet1

# Construct the URL for reading the sheet
# Use the gviz/tq endpoint to export the sheet as a CSV
url = f'https://docs.google.com/spreadsheets/d/{file_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}'

# Read the data into a pandas DataFrame
df = pd.read_csv(url)

# Display the first few rows of the DataFrame
df

# 2-Precipitation

## 2.1 Multiple AOIs (Iterative)

In [ ]:
# Create the Dictionary
dict_precipitation = {
    'Monthly': {
        'dataset': 'ECMWF/ERA5_LAND/MONTHLY_AGGR',
        'band' : 'total_precipitation_sum',
        'lag_days'  : 365,
    },
    'Daily': {
        'dataset': 'ECMWF/ERA5_LAND/DAILY_AGGR',
        'band': 'total_precipitation_sum',
        'lag_days'  : 30,
    },
    'Hourly': {
        'dataset': 'ECMWF/ERA5_LAND/HOURLY',
        'band': 'total_precipitation_hourly',
        'lag_days'  : 1
    }
}

In [ ]:
timescale = 'Daily' # Monthly # Daily # Hourly
start_date = '2024-3-1' # 2024-02-11 #start of the checking period  high-flow --> 2023-11-10     low-flow --> 2024-7-
start_date_datetime_obj = datetime.datetime.strptime(start_date, '%Y-%m-%d')
current_target_datetime_obj = datetime.datetime.strptime(start_date, '%Y-%m-%d') + datetime.timedelta(days=dict_precipitation[timescale]['lag_days'])

long_months = [1, 3, 5, 7, 8, 10, 12]

if timescale == 'Daily' and start_date_datetime_obj.month in long_months:
    lag_days = 31
else:
    # If not 'Daily' or not a 'long month' for 'Daily', use the default lag_days from dict_precipitation
    # This ensures lag_days is always defined for the chosen timescale.
    lag_days = dict_precipitation[timescale]['lag_days']

target_date_str = (start_date_datetime_obj + datetime.timedelta(days=lag_days)).strftime('%Y-%m-%d') #end of the checking period, formatted

print(f"Target Date Start: {start_date_datetime_obj}\nTarget Date End: {target_date_str}")

In [ ]:
all_aoi_dfs = []

for index, row in df.iterrows():
    event_number = row['Event_Number']
    area_of_interest = row['AOI']
    #target_date_str = str(row['Target_Date'])
    center_long = row['Center_LONG']
    center_lat = row['Center_LAT']

    # Skip rows with NaN coordinates or missing AOI/Target_Date
    if pd.isna(center_long) or pd.isna(center_lat) or pd.isna(area_of_interest) or pd.isna(target_date_str):
        print(f"Skipping Event {event_number} due to missing data (AOI, coordinates, or Target_Date).")
        continue

    try:
        # Area of Interest Definition
        current_target_date_dt = datetime.datetime.strptime(target_date_str, '%Y-%m-%d')

        # Define the geometry for the current AOI
        geom_current_aoi = ee.Geometry.Point([center_long, center_lat])

        # Generate the dates for the precipitation collection window
        precipitation_start_date = (current_target_date_dt - datetime.timedelta(days=lag_days)).strftime('%Y-%m-%d')
        precipitation_end_date = current_target_date_dt.strftime('%Y-%m-%d')

        print(f"\nProcessing AOI: {area_of_interest} (Event: {int(event_number)})\n  Dataset: {dict_precipitation[timescale]['dataset']}\n  Band: {dict_precipitation[timescale]['band']}\n  Start Date: {precipitation_start_date}\n  End Date: {precipitation_end_date}")

        # Load the ERA5-Land time series data
        precipitation_collection_aoi = (
            ee.ImageCollection(dict_precipitation[timescale]['dataset'])
            .filterDate(precipitation_start_date, precipitation_end_date)
            .select(dict_precipitation[timescale]['band'])
        )

        # Re-define the extract_feature function for clarity within the loop context
        def extract_feature_for_aoi(image):
            mean_feature = image.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=geom_current_aoi,
                scale=1000,
                maxPixels=1e10
            ).get(dict_precipitation[timescale]['band'])
            timestamp = image.get('system:time_start')
            return ee.Feature(None, {'precipitation': mean_feature, 'timestamp': timestamp})

        # Map the function over the image collection and convert to a list of features
        precipitation_features_aoi = precipitation_collection_aoi.map(extract_feature_for_aoi).getInfo()

        # Convert the list of features to a pandas DataFrame
        precipitation_data_aoi = []
        for feature in precipitation_features_aoi['features']:
            precipitation_data_aoi.append(feature['properties'])

        df_precipitation_aoi = pd.DataFrame(precipitation_data_aoi)

        # Convert timestamp to datetime
        if not df_precipitation_aoi.empty:
            df_precipitation_aoi['timestamp'] = pd.to_datetime(df_precipitation_aoi['timestamp'], unit='ms')
            df_precipitation_aoi['AOI'] = area_of_interest
            df_precipitation_aoi['Event_Number'] = int(event_number)
            all_aoi_dfs.append(df_precipitation_aoi)
        else:
            print(f"No precipitation data found for {area_of_interest} ({precipitation_start_date} to {precipitation_end_date}).")

    except Exception as e:
        print(f"Error processing Event {event_number} ({area_of_interest}): {e}")

In [ ]:
if all_aoi_dfs:
    combined_precipitation_df = pd.concat(all_aoi_dfs, ignore_index=True)
    print("Combined Precipitation DataFrame:")
    display(combined_precipitation_df.head())
    display(combined_precipitation_df.tail())
else:
    print("No data collected for any AOI.")
    combined_precipitation_df = pd.DataFrame()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if not combined_precipitation_df.empty:
    plt.figure(figsize=(15, 8))
    for aoi_name in combined_precipitation_df['AOI'].unique():
        aoi_df = combined_precipitation_df[combined_precipitation_df['AOI'] == aoi_name]
        # Only plot if there are valid precipitation values for this AOI
        if not aoi_df['precipitation'].isnull().all():
            plt.plot(aoi_df['timestamp'], aoi_df['precipitation'], label=aoi_name)
        else:
            print(f"Skipping {aoi_name} from the plot as it contains no valid precipitation data.")

    plt.title(f'{timescale} Precipitation Time Series for Multiple AOIs')
    plt.xlabel('Time')
    plt.ylabel('Precipitation (m)')

    # Define a dictionary for tick format based on timescale
    tick_format_dict = {
        'Monthly': {'locator': mdates.MonthLocator(), 'formatter': mdates.DateFormatter('%Y-%m')},
        'Daily': {'locator': mdates.DayLocator(), 'formatter': mdates.DateFormatter('%Y-%m-%d')},
        'Hourly': {'locator': mdates.HourLocator(), 'formatter': mdates.DateFormatter('%H:%M')}
    }

    ax = plt.gca()
    # Apply the appropriate locator and formatter based on the 'timescale' variable
    if timescale in tick_format_dict:
        ax.xaxis.set_major_locator(tick_format_dict[timescale]['locator'])
        ax.xaxis.set_major_formatter(tick_format_dict[timescale]['formatter'])
    else:
        # Default to monthly if timescale is not recognized
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

    plt.xticks(rotation=45)
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("Cannot visualize: Combined DataFrame is empty.")

In [ ]:
if not combined_precipitation_df.empty:
    print("Maximum Precipitation Dates for Each AOI:")
    for aoi_name in combined_precipitation_df['AOI'].unique():
        aoi_df = combined_precipitation_df[combined_precipitation_df['AOI'] == aoi_name]
        # Check if there are any non-NaN precipitation values for the current AOI
        if not aoi_df['precipitation'].isnull().all():
            max_precipitation_aoi = aoi_df['precipitation'].max()
            # Filter out NaNs from the precipitation column before finding the corresponding timestamp
            valid_precipitation_df = aoi_df.dropna(subset=['precipitation'])
            if not valid_precipitation_df.empty:
                max_precipitation_date_aoi = valid_precipitation_df.loc[valid_precipitation_df['precipitation'] == max_precipitation_aoi, 'timestamp'].values[0]
                print(f"  AOI: {aoi_name}\n    Maximum Precipitation: {max_precipitation_aoi:.4f} m\n    Date: {pd.to_datetime(max_precipitation_date_aoi).date()}\n")
            else:
                print(f"  AOI: {aoi_name}\n    No valid precipitation data to determine maximum.\n")
        else:
            print(f"  AOI: {aoi_name}\n    No valid precipitation data to determine maximum.\n")
else:
    print("Cannot determine maximum precipitation: Combined DataFrame is empty.")

In [ ]:
if not combined_precipitation_df.empty:
    output_filename = 'combined_precipitation_data.csv'
    combined_precipitation_df.to_csv(output_filename, index=False)
    print(f"Combined precipitation data exported to {output_filename}")
else:
    print("Cannot export: Combined DataFrame is empty.")

## 2.2 Muliple AOIs (Automatic)


In [ ]:
Event = '2' #1 #2
print(f"Plotting Event: {Event}")

In [ ]:
def get_era5_precipitation_data(timescale, geom, precipitation_start_date, precipitation_end_date):
    """
    Fetches and processes ERA5-Land precipitation data from Google Earth Engine.

    Args:
        timescale (str): The desired timescale ('Monthly', 'Daily', 'Hourly').
        geom (ee.Geometry): The Earth Engine geometry (e.g., ee.Geometry.Point) for the AOI.
        precipitation_start_date (str): The start date for filtering data (YYYY-MM-DD).
        precipitation_end_date (str): The end date for filtering data (YYYY-MM-DD).

    Returns:
        pd.DataFrame: A DataFrame with 'precipitation' values and 'timestamp' for the AOI.
                      Returns an empty DataFrame if no data is found.
    """

    # 2. Inside this function, define the dict_precipitation dictionary
    dict_precipitation = {
        'Monthly': {
            'dataset': 'ECMWF/ERA5_LAND/MONTHLY_AGGR',
            'band': 'total_precipitation_sum',
            'lag_days': 365,
        },
        'Daily': {
            'dataset': 'ECMWF/ERA5_LAND/DAILY_AGGR',
            'band': 'total_precipitation_sum',
            'lag_days': 30,
        },
        'Hourly': {
            'dataset': 'ECMWF/ERA5_LAND/HOURLY',
            'band': 'total_precipitation_hourly',
            'lag_days': 1
        }
    }

    # Ensure the requested timescale exists in the dictionary
    if timescale not in dict_precipitation:
        raise ValueError(f"Invalid timescale: {timescale}. Must be 'Monthly', 'Daily', or 'Hourly'.")

    # 3. Construct an ee.ImageCollection
    precipitation_collection = (
        ee.ImageCollection(dict_precipitation[timescale]['dataset'])
        .filterDate(precipitation_start_date, precipitation_end_date)
        .filterBounds(geom) # Filter by geometry to reduce data for export
        .select(dict_precipitation[timescale]['band'])
    )

    # 4. Define an inner helper function, extract_feature
    def extract_feature(image):
        """Extracts mean precipitation and timestamp from an image."""
        mean_feature = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=dict_precipitation[timescale].get('scale', 1000), # Use default 1000m or specific scale if defined
            maxPixels=1e10
        ).get(dict_precipitation[timescale]['band'])

        timestamp = image.get('system:time_start')

        return ee.Feature(None, {
            'precipitation': mean_feature,
            'timestamp': timestamp
        })

    # 5. Apply the extract_feature function and retrieve results
    precipitation_features = precipitation_collection.map(extract_feature).getInfo()

    # 6. Convert the list of features into a pandas DataFrame
    precipitation_data = []
    for feature in precipitation_features['features']:
        properties = feature['properties']
        # Handle cases where reduceRegion might return null for precipitation
        if properties['precipitation'] is not None:
            precipitation_data.append(properties)

    df_precipitation = pd.DataFrame(precipitation_data)

    # 7. Convert the 'timestamp' column in the DataFrame to datetime objects
    if not df_precipitation.empty:
        df_precipitation['timestamp'] = pd.to_datetime(df_precipitation['timestamp'], unit='ms')

    # 8. Return the processed pandas DataFrame
    return df_precipitation

print("The `get_era5_precipitation_data` function has been defined.")

In [ ]:
import datetime

# The dict_precipitation is internally defined in `get_era5_precipitation_data`
# but we need it here to determine the date ranges for each timescale.
# Copying it here for clarity or assuming it's available globally if needed for calculations.
dict_precipitation = {
    'Monthly': {
        'dataset': 'ECMWF/ERA5_LAND/MONTHLY_AGGR',
        'band' : 'total_precipitation_sum',
        'lag_days'  : 365,
    },
    'Daily': {
        'dataset': 'ECMWF/ERA5_LAND/DAILY_AGGR',
        'band': 'total_precipitation_sum',
        'lag_days'  : 30,
    },
    'Hourly': {
        'dataset': 'ECMWF/ERA5_LAND/HOURLY',
        'band': 'total_precipitation_hourly',
        'lag_days'  : 1
    }
}
date_dict = {
    '1': {
        'Monthly': '2023-11-1',
        'Daily': '2024-11-1',
        'Hourly': '2023-11-30'  #Event 1: 2023-11-30
        },
    '2': {
        'Monthly': '2023-11-1',
        'Daily': '2024-11-1',
        'Hourly': '2023-11-30'  #Event 2: 2024-3-11
        },
    }

# 1. Initialize precipitation_data_dfs as an empty dictionary
precipitation_data_dfs = {}

# 2. Initialize temp_aoi_data to store individual AOI DataFrames before concatenation
temp_aoi_data = {'Monthly': [], 'Daily': [], 'Hourly': []}

timescales_to_process = ['Monthly', 'Daily', 'Hourly']

# 3. Iterate through each row of the df DataFrame
for index, row in df.iterrows():
    aoi_name = row['AOI']
    center_long = row['Center_LONG']
    center_lat = row['Center_LAT']

    # Skip rows with NaN coordinates or missing AOI
    if pd.isna(center_long) or pd.isna(center_lat) or pd.isna(aoi_name):
        print(f"Skipping row {index} due to missing data (AOI or coordinates).")
        continue

    # 5. Create an ee.Geometry.Point named geom_current_aoi
    geom_current_aoi = ee.Geometry.Point([center_long, center_lat])

    print(f"\nProcessing AOI: {aoi_name}")

    # 6. For each timescale in timescales_to_process
    for ts in timescales_to_process:
        # Convert target_date string to datetime object for calculations
        start_date = date_dict[Event][ts] # This is the target_date (end of the period) for the current timescale
        current_target_datetime_obj = datetime.datetime.strptime(start_date, '%Y-%m-%d') + datetime.timedelta(days=dict_precipitation[ts]['lag_days'])
        print(f"Target datetime object for {ts}: {current_target_datetime_obj}")
        print(f"  Fetching {ts} data...")
        try:
            # a. Determine the timescale_lag_days
            timescale_lag_days = dict_precipitation[ts]['lag_days']

            # b. Calculate ts_precipitation_start_date and ts_precipitation_end_date
            # The query window will be from (target_date - timescale_lag_days) to target_date.
            ts_precipitation_start_date = (current_target_datetime_obj - datetime.timedelta(days=timescale_lag_days)).strftime('%Y-%m-%d')
            ts_precipitation_end_date = current_target_datetime_obj.strftime('%Y-%m-%d') # The end date for filtering

            # c. Call the get_era5_precipitation_data function
            df_ts_precipitation = get_era5_precipitation_data(
                timescale=ts,
                geom=geom_current_aoi,
                precipitation_start_date=ts_precipitation_start_date,
                precipitation_end_date=ts_precipitation_end_date
            )

            if not df_ts_precipitation.empty:
                # d. Add a new column 'AOI' to it
                df_ts_precipitation['AOI'] = aoi_name
                # e. Append this df_ts_precipitation to temp_aoi_data
                temp_aoi_data[ts].append(df_ts_precipitation)
                print(f"    Successfully retrieved {len(df_ts_precipitation)} records for {ts}.")
            else:
                print(f"    No data retrieved for {ts} for AOI {aoi_name} ({ts_precipitation_start_date} to {ts_precipitation_end_date}).")

        except Exception as e:
            print(f"    Error retrieving {ts} data for AOI {aoi_name}: {e}")

# 7. After the loops, concatenate all DataFrames for each timescale
for ts in timescales_to_process:
    if temp_aoi_data[ts]: # Check if the list is not empty
        precipitation_data_dfs[ts] = pd.concat(temp_aoi_data[ts], ignore_index=True)
        print(f"\nCombined DataFrame for {ts}:")
        display(precipitation_data_dfs[ts].head())
    else:
        print(f"\nNo data to combine for {ts}.")

# 8. Print a confirmation message
print("\nData retrieval for all AOIs and timescales is complete.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

print(f"Plotting Event: {Event}")

# Set a global font size for better readability
plt.rcParams.update({'font.size': 15})

# 1. Create a figure and a set of subplots
fig, axs = plt.subplots(nrows=3, figsize=(14, 15), sharex=False)

# 2. Define a dictionary for tick format based on timescale
tick_format_dict = {
    'Monthly': {'locator': mdates.MonthLocator(), 'formatter': mdates.DateFormatter('%Y-%m')},
    'Daily': {'locator': mdates.DayLocator(interval=5), 'formatter': mdates.DateFormatter('%Y-%m-%d')},
    'Hourly': {'locator': mdates.HourLocator(interval=6), 'formatter': mdates.DateFormatter('%H:%M')}
}

# Ensure the timescales are in the desired order for plotting
timescales_to_process = ['Monthly', 'Daily', 'Hourly']

# 3. Iterate through the timescales and their corresponding axes
for i, ts in enumerate(timescales_to_process):
    ax = axs[i]

    if ts in precipitation_data_dfs and not precipitation_data_dfs[ts].empty:
        df_ts_combined = precipitation_data_dfs[ts]

        # Plot each AOI's time series on the current subplot
        for aoi_name in df_ts_combined['AOI'].unique():
            aoi_df = df_ts_combined[df_ts_combined['AOI'] == aoi_name]
            if not aoi_df['precipitation'].isnull().all():
                ax.plot(aoi_df['timestamp'], aoi_df['precipitation'], label=aoi_name)
            else:
                print(f"Skipping {aoi_name} from {ts} plot as it contains no valid precipitation data.")

        ax.set_title(f'{ts} Precipitation Time Series for Multiple AOIs')
        ax.set_ylabel('Precipitation (m)')
        ax.legend(title='AOI')

        # Apply the appropriate locator and formatter
        ax.xaxis.set_major_locator(tick_format_dict[ts]['locator'])
        ax.xaxis.set_major_formatter(tick_format_dict[ts]['formatter'])

        # Rotate x-axis tick labels for better readability
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

        ax.grid(True)
    else:
        ax.set_title(f'{ts} Precipitation Time Series - No Data')
        ax.set_ylabel('Precipitation (m)')
        ax.text(0.5, 0.5, 'No data available', horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)

plt.xlabel('Time') # Set common x-label for the whole figure if desired, or let individual subplots manage
plt.tight_layout()
plt.show()

In [ ]:
for timescale, df_precipitation in precipitation_data_dfs.items():
    if not df_precipitation.empty:
        output_filename = f"{timescale.lower()}_precipitation_data.csv"
        df_precipitation.to_csv(output_filename, index=False)
        print(f"Exported {timescale} precipitation data to {output_filename}")
    else:
        print(f"No data to export for {timescale}.")